In [1]:
import os, random, re
import numpy as np, pandas as pd

seed = 12345
random.seed(seed)
np.random.seed(seed)

import tensorflow as tf

tf.random.set_seed(seed)

print(os.listdir("../input"))




2026-05-05 22:27:44.549010: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778020064.561990      56 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778020064.565956      56 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 22:27:44.582069: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

['train.tsv.zip', 'movie-review-sentiment-analysis-kernels-only', 'sampleSubmission.csv', 'test.tsv.zip', 'description.md', 'sampleSubmission.csv.zip', 'test.tsv', 'train.tsv']


In [2]:
train = pd.read_csv("../input/train.tsv", sep="\t")
test = pd.read_csv("../input/test.tsv", sep="\t")
sampleSubmission = pd.read_csv("../input/sampleSubmission.csv")
print(train.head(3))




   PhraseId  SentenceId               Phrase  Sentiment
0     24144        1097  through every frame          2
1     85111        4402    onscreen presence          2
2     75999        3900     into dysfunction          2


In [3]:
def clean_str(string):
    string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", string)
    string = re.sub(r"\'s", " 's", string)
    string = re.sub(r"\'ve", " 've", string)
    string = re.sub(r"n\'t", " n't", string)
    string = re.sub(r"\'re", " 're", string)
    string = re.sub(r"\'d", " 'd", string)
    string = re.sub(r"\'ll", " 'll", string)
    string = re.sub(r",", " , ", string)
    string = re.sub(r"!", " ! ", string)
    string = re.sub(r"\(", " \( ", string)
    string = re.sub(r"\)", " \) ", string)
    string = re.sub(r"\?", " \? ", string)
    string = re.sub(r"\s{2,}", " ", string)
    return string.strip().lower()


phrases = [clean_str(s) for s in train["Phrase"]]




In [4]:
from tensorflow.keras.utils import to_categorical

Y = to_categorical(train["Sentiment"].values)
print(Y[155:165])
print(train["Sentiment"].values[155:165])




[[0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]]
[2 2 2 2 3 4 3 3 2 2]


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

t = Tokenizer()
t.fit_on_texts(phrases)
vocab_size = len(t.word_index) + 1

X_seq = t.texts_to_sequences(phrases)
max_length = max(len(p.split()) for p in phrases)
X = pad_sequences(X_seq, maxlen=max_length, padding="post")
print("X shape:", X.shape)




X shape: (109242, 52)


In [6]:
from tensorflow.keras.layers import (
    Embedding,
    Flatten,
    Dense,
    Conv1D,
    SpatialDropout1D,
    MaxPooling1D,
    concatenate,
    Input,
    Dropout,
)
from tensorflow.keras.models import Sequential, Model


def model1(output_dim=8, max_length=50, y_dim=5):
    model = Sequential()
    model.add(Embedding(vocab_size, output_dim, input_length=max_length))
    model.add(Flatten())
    model.add(Dense(y_dim, activation="softmax"))
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["acc"])
    return model


def model2(output_dim=8, max_length=50, y_dim=5, num_filters=5, filter_sizes=[3]):
    model = Sequential()
    model.add(Embedding(vocab_size, output_dim, input_length=max_length))
    model.add(SpatialDropout1D(0.2))
    model.add(
        Conv1D(
            num_filters, kernel_size=filter_sizes[0], padding="valid", activation="relu"
        )
    )
    model.add(
        MaxPooling1D(
            pool_size=max_length - filter_sizes[0] + 1, strides=1, padding="valid"
        )
    )
    model.add(Flatten())
    model.add(Dense(y_dim, activation="softmax"))
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["acc"])
    return model


def model3(output_dim=8, max_length=50, y_dim=5, num_filters=5, filter_sizes=[3, 5]):
    embed_input = Input(shape=(max_length,))
    x = Embedding(vocab_size, output_dim, input_length=max_length)(embed_input)
    pooled_outputs = []
    for sz in filter_sizes:
        conv = Conv1D(num_filters, kernel_size=sz, padding="valid", activation="relu")(
            x
        )
        conv = MaxPooling1D(pool_size=max_length - sz + 1, strides=1, padding="valid")(
            conv
        )
        pooled_outputs.append(conv)
    merged = concatenate(pooled_outputs)
    x = Flatten()(merged)
    x = Dropout(0.2)(x)
    predictions = Dense(y_dim, activation="softmax")(x)
    model = Model(inputs=embed_input, outputs=predictions)
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["acc"])
    return model


# instantiate the CNN model (you may switch to model1/model2 if desired)
model = model3(output_dim=8, max_length=max_length, y_dim=5, filter_sizes=[3, 4, 5])




/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-05 22:27:53.621191: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778020073.621630      56 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.25, random_state=seed
)




In [8]:
epochs = 20
batch_size = 32

model.fit(
    X_train,
    Y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_val, Y_val),
    verbose=1,
)

loss, accuracy = model.evaluate(X_val, Y_val, verbose=0)
print(f"Validation Accuracy: {accuracy*100:.2f}%")




Epoch 1/20


I0000 00:00:1778020075.059415     109 service.cc:148] XLA service 0x7f71100046f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778020075.059449     109 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-05 22:27:55.082790: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778020075.190914     109 cuda_dnn.cc:529] Loaded cuDNN version 90300


 126/2561 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - acc: 0.5124 - loss: 1.4821

I0000 00:00:1778020076.768148     109 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2561/2561 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - acc: 0.5196 - loss: 1.2749 - val_acc: 0.5830 - val_loss: 1.0376
Epoch 2/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - acc: 0.6087 - loss: 1.0017 - val_acc: 0.6230 - val_loss: 0.9303
Epoch 3/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - acc: 0.6500 - loss: 0.8755 - val_acc: 0.6339 - val_loss: 0.9146
Epoch 4/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - acc: 0.6732 - loss: 0.8153 - val_acc: 0.6380 - val_loss: 0.9152
Epoch 5/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - acc: 0.6881 - loss: 0.7784 - val_acc: 0.6406 - val_loss: 0.9221
Epoch 6/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - acc: 0.6983 - loss: 0.7448 - val_acc: 0.6422 - val_loss: 0.9346
Epoch 7/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - acc: 0.7069 - loss: 0.7269 - val_acc: 0.6428 - val_loss: 0.9449
Epoch 8/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step - acc: 0.7175 - loss: 0.7045 - val_acc: 0.6431 - val_loss: 0.9612
Epoch 9/20
2561/2561 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/st

In [9]:
test_phrases = [clean_str(s) for s in test["Phrase"]]
X_test_seq = t.texts_to_sequences(test_phrases)
X_test = pad_sequences(X_test_seq, maxlen=max_length, padding="post")




In [10]:
preds = model.predict(X_test, verbose=1).argmax(axis=-1)
sampleSubmission["Sentiment"] = preds
sampleSubmission.to_csv("sub_cnn3.csv", index=False)
print("Submission saved to sub_cnn3.csv")

1464/1464 ━━━━━━━━━━━━━━━━━━━━ 1s 756us/step
Submission saved to sub_cnn3.csv
